# Gemma 4 API Host - Nexus Claire 🧠
This notebook turns a Google Colab T4 instance into a fully functional, OpenAI-compatible API endpoint running **Gemma 4**, exposed over the internet via `pyngrok`.

## Setup Instructions
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Get your free [ngrok Auth Token](https://dashboard.ngrok.com/get-started/your-authtoken) and paste it in the cell below.
3. Optional: Get your [HuggingFace Token](https://huggingface.co/settings/tokens) (required for some models like larger variants, but Gemma 4 open weights might just work. We'll use the unsloth pre-quantized or base models).
4. Run all cells (`Ctrl+F9`).

In [ ]:
!pip install pyngrok -q
!pip install vllm -q

In [ ]:
import os

# 🛑 STOP AND FILL THESE OUT 🛑
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"
HF_TOKEN = "YOUR_HF_TOKEN_HERE_OPTIONAL"

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE_OPTIONAL":
    from huggingface_hub import login
    login(HF_TOKEN)

# We use E4B as it balances incredible reasoning with the 16GB VRAM limit on Colab T4
MODEL_ID = "google/gemma-4-E4B-it" 

os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTH_TOKEN

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Connect to ngrok
if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN_HERE":
    print("❌ ERROR: Please add your NGROK_AUTH_TOKEN in the previous cell! ❌")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000).public_url

    print("\n======================================================")
    print(f"🟢 YOUR API BASE URL: {public_url}/v1")
    print("======================================================\n")
    
    print("Starting vLLM server... This will take a few minutes to download the model and allocate VRAM.")
    
    # Launch the vLLM OpenAI compatible server
    # We constrain the max_model_len to fit within the T4's 16GB memory alongside the E4B weights
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_ID,
        "--host", "0.0.0.0",
        "--port", "8000",
        "--max-model-len", "8192",   # Reduces KV cache size so it fits perfectly
        "--enable-auto-tool-choice", # Required for Gemma 4 tool use
        "--tool-call-parser", "gemma4",
        "--reasoning-parser", "gemma4"
    ]
    
    # Run the server and stream logs
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        print("Terminating server...")
        process.terminate()
        ngrok.kill()